# Learning costs (structured perceptron)

_Artificial Intelligence — more_

**Score whole structures. When you pick the wrong one, reward the right one and punish your pick.**

Search needs costs (or scores) on actions. Where do those numbers come from? We can learn them.

     The structured perceptron scores a whole output (a path, a parse, a labeling) and fixes its weights when it picks the wrong one.

---

This notebook is a practice scaffold for the **Learning costs (structured perceptron)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Encode each candidate structure as a feature vectorThe structured perceptron scores a *whole output* — here a path through a graph — by a dot product `w · phi`, where `phi` counts how often each part (edge) is used. We set up two candidate paths over four possible edges `[e1, e2, e3, e4]`: the **true** path uses edges 1 and 2, and an **impostor** path uses edges 3 and 4. We start the weights at zero, so initially every path scores `0`.

In [ ]:
import numpy as np

# features = edge-usage counts over 4 edges [e1, e2, e3, e4].
# True path uses e1,e2 ; one impostor path uses e3,e4.
phi_true = np.array([1, 1, 0, 0])
phi_wrong = np.array([0, 0, 1, 1])

# one weight per edge; start at zero so both paths score 0 to begin with
w = np.zeros(4)
print("starting weights:", w.tolist())

### Step 2 — Predict, then correct only on a mistakeEach step we score both candidates and **predict** the highest-scoring one. On a tie we deliberately prefer the impostor, which counts as a mistake so learning has something to fix. When the prediction is wrong, the structured perceptron update `w ← w + phi_true − phi_hat` *rewards* the features of the correct path and *punishes* the features of the path we wrongly picked.

In [ ]:
for step in range(3):
    # predict the highest-scoring path; on a tie prefer the impostor (counts as a mistake)
    s_true = w @ phi_true
    s_wrong = w @ phi_wrong
    phi_hat = phi_wrong if s_wrong >= s_true else phi_true

    if not np.array_equal(phi_hat, phi_true):     # mistake -> structured perceptron update
        w = w + phi_true - phi_hat                # reward the truth, punish our pick

    print(f"step {step}: w={w.tolist()}  true={w @ phi_true:.0f}  wrong={w @ phi_wrong:.0f}")

### Step 3 — Check the learned weightsAfter the updates, the weights should make the true path outscore the impostor. We print the final weight vector and confirm that `w · phi_true` is now strictly greater than `w · phi_wrong` — the perceptron has learned costs that prefer the correct structure.

In [ ]:
print("final weights:", w.tolist())
print("true path wins?", (w @ phi_true) > (w @ phi_wrong))

## Visualize the data & results

_POS-tagging 'dogs chase cats': does one perceptron update make the correct NOUN VERB NOUN tagging win?_

### Step 1 — Set up a real tagging example and its featuresWe move from abstract edges to a real structured-prediction task: part-of-speech tagging the sentence *"dogs chase cats"*. The **gold** tagging is `NOUN VERB NOUN`; a plausible **wrong** tagging is `NOUN NOUN NOUN`. The feature function turns a tagging into counts of two kinds of parts — *emission* features (a word paired with its tag) and *transition* features (one tag following another) — exactly the pieces the weights will score.

In [ ]:
import matplotlib.pyplot as plt

# POS tagging the real sentence "dogs chase cats". Gold = NOUN VERB NOUN.
sent = ["dogs", "chase", "cats"]
gold = ["NOUN", "VERB", "NOUN"]
wrong = ["NOUN", "NOUN", "NOUN"]

def feats(words, tags):
    f = {}
    for w, t in zip(words, tags):                 # emission features: (word, tag)
        f[("emit", w, t)] = f.get(("emit", w, t), 0) + 1
    for i in range(1, len(tags)):                 # transition features: (prev tag, tag)
        k = ("trans", tags[i-1], tags[i])
        f[k] = f.get(k, 0) + 1
    return f

def score(w, f):
    return sum(w.get(k, 0.0) * v for k, v in f.items())

fg = feats(sent, gold)
fw = feats(sent, wrong)
print("gold features :", dict(fg))
print("wrong features:", dict(fw))

### Step 2 — Apply one perceptron updateWith zero weights both taggings score `0`. We apply a single structured-perceptron update: add the gold tagging's features and subtract the predicted (wrong) tagging's features. We record the scores **before** (both `0`) and **after** the update so we can see exactly what one correction does.

In [ ]:
w = {}
before = [score(w, fg), score(w, fw)]             # both 0 at the start

for k, v in fg.items():
    w[k] = w.get(k, 0.0) + v                       # +gold features (reward the truth)
for k, v in fw.items():
    w[k] = w.get(k, 0.0) - v                       # -predicted features (punish our pick)

after = [score(w, fg), score(w, fw)]              # +3 for gold vs -5 for wrong
print("scores before update:", before)
print("scores after update :", after)

### Step 3 — Plot the scores after the updateThe bar chart shows the score of each tagging after a single update. The gold `NOUN VERB NOUN` is now clearly positive while the wrong `NOUN NOUN NOUN` is negative — one perceptron correction was enough to make the correct structure win.

In [ ]:
labels = ["correct: NOUN VERB NOUN", "wrong: NOUN NOUN NOUN"]
colors = ["#7ee787", "#ff7b72"]

fig, ax = plt.subplots()
bars = ax.bar(labels, after, color=colors)
for b, v in zip(bars, after):
    va = "bottom" if v >= 0 else "top"
    ax.text(b.get_x() + b.get_width()/2, v, str(int(v)), ha="center", va=va)
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Score of tagging 'dogs chase cats' before vs after one update")
plt.show()